# Arrange patch products by `pXXXXX`

Takes one **input root** that contains (by type):

```
<input_root>/
  fabdem_10m_bicubic/   ->  {n}_FABDEM_10mbicubic_pXXXXX.tif
  fabdem_30m/           ->  {n}_FABDEM_30m_pXXXXX.tif
  lidar_shp/            ->  {n}_LiDAR_pXXXXX.shp (+ .shx/.dbf/.prj/.cpg)
  gt_visual/            ->  long HMA name ending with _pXXXXX.tif
```

and writes **one folder per patch** under the output dir:

```
<output_root>/
  p00072/
    1284_FABDEM_10mbicubic_p00072.tif
    1284_FABDEM_30m_p00072.tif
    1284_LiDAR_p00072.shp  (+ sidecars)
    00000_HMA_DTM_10m_EGM08_p00072.tif
```

### HMA rename rule

**From** (example):

`00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.tif`

**To**:

`00000_HMA_DTM_10m_EGM08_p00072.tif`


## 1. Imports


In [1]:
import os
import re
import shutil
from collections import defaultdict
from pathlib import Path

print("imports OK")


imports OK


## 2. Paths & options

Set `INPUT_ROOT` to the folder that contains the four type-folders (as in your screenshot).  
Set `OUTPUT_ROOT` to where per-patch folders should be created.


In [2]:
INPUT_ROOT = r"D:\Vaibhav\dem-lidar\inference_old"
GT_ROOT    = r"D:\Vaibhav\dem-lidar\gt_visual_256"

# Where pXXXXX folders will be created
OUTPUT_ROOT = r"D:\Vaibhav\dem-lidar\inference"

# Subfolder names under INPUT_ROOT (change if yours differ)
DIR_FAB10  = "fabdem_10m_bicubic"
DIR_FAB30  = "fabdem_30m"
DIR_LIDAR  = "lidar_shp"

# True  = copy files (keep originals)
# False = move files into patch folders
COPY_MODE = True

# If a patch is missing one of the 4 products, still create the folder with what exists
ALLOW_PARTIAL = True

print("INPUT_ROOT :", INPUT_ROOT)
print("GT_ROOT    :", GT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("COPY_MODE  :", COPY_MODE)
print("ALLOW_PARTIAL:", ALLOW_PARTIAL)


INPUT_ROOT : D:\Vaibhav\dem-lidar\inference_old
GT_ROOT    : D:\Vaibhav\dem-lidar\gt_visual_256
OUTPUT_ROOT: D:\Vaibhav\dem-lidar\inference
COPY_MODE  : True
ALLOW_PARTIAL: True


## 3. Helpers — parse serials & simplify HMA names


In [3]:
# Match serial ..._p00072 at end of name
_SERIAL_RE = re.compile(r"_p(?P<serial>\d+)\b", re.IGNORECASE)

# Product patterns (stem without extension)
_FAB10_RE = re.compile(r"^(?P<n>\d+)_FABDEM_10mbicubic_p(?P<serial>\d+)$", re.I)
_FAB30_RE = re.compile(r"^(?P<n>\d+)_FABDEM_30m_p(?P<serial>\d+)$", re.I)
_LIDAR_RE = re.compile(r"^(?P<n>\d+)_LiDAR_p(?P<serial>\d+)$", re.I)
# GT visual: {photons}_{anything}_p{idx}
_GT_RE = re.compile(r"^(?P<n>\d+)_(?P<body>.+)_p(?P<serial>\d+)$", re.I)

# Shapefile sidecar extensions to keep together
SHP_SIDECARS = (".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx", ".xml")


def extract_serial(name: str):
    """Return p00072-style string from any filename, or None."""
    m = _SERIAL_RE.search(name)
    if not m:
        return None
    return f"p{int(m.group('serial')):05d}"


def simplify_hma_name(filename):
    stem, ext = os.path.splitext(filename)

    m = re.search(
        r"^(\d+)_.*?_DTM_10m_EGM08_p(\d+)$",
        stem,
        re.IGNORECASE,
    )

    if not m:
        return filename

    photons = int(m.group(1))
    patch = int(m.group(2))

    return f"{photons:05d}_HMA_DTM_10m_EGM08_p{patch:05d}{ext.lower()}"


def list_files(folder: Path):
    if not folder.is_dir():
        return []
    return [p for p in folder.iterdir() if p.is_file()]


def transfer(src: Path, dst: Path, copy_mode: bool = True):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() == dst.resolve():
        return "skip-same"
    if copy_mode:
        shutil.copy2(src, dst)
        return "copy"
    if dst.exists():
        dst.unlink()
    shutil.move(str(src), str(dst))
    return "move"


print("helpers ready")
print("example HMA rename:")
ex = "00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.tif"
print(" ", ex)
print(" ->", simplify_hma_name(ex))


helpers ready
example HMA rename:
  00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.tif
 -> 00000_HMA_DTM_10m_EGM08_p00072.tif


## 4. Index files by `pXXXXX`


In [4]:
input_root = Path(INPUT_ROOT)
output_root = Path(OUTPUT_ROOT)

folders = {
    "fab10": input_root / DIR_FAB10,
    "fab30": input_root / DIR_FAB30,
    "lidar": input_root / DIR_LIDAR,
    "gt":    Path(GT_ROOT),
}

print("Checking input folders:")
for k, p in folders.items():
    exists = p.is_dir()
    n = len(list_files(p)) if exists else 0
    print(f"  [{'OK' if exists else 'MISSING'}] {k:5s}  {p}  ({n} files)")

# patches[serial] = paths for each product
patches = defaultdict(lambda: {"fab10": None, "fab30": None, "lidar": None, "gt": None})

# FABDEM 10 m
for p in list_files(folders["fab10"]):
    if p.suffix.lower() not in (".tif", ".tiff"):
        continue
    m = _FAB10_RE.match(p.stem)
    serial = f"p{int(m.group('serial')):05d}" if m else extract_serial(p.name)
    if serial:
        patches[serial]["fab10"] = p

# FABDEM 30 m
for p in list_files(folders["fab30"]):
    if p.suffix.lower() not in (".tif", ".tiff"):
        continue
    m = _FAB30_RE.match(p.stem)
    serial = f"p{int(m.group('serial')):05d}" if m else extract_serial(p.name)
    if serial:
        patches[serial]["fab30"] = p

# LiDAR shapefiles (index .shp only; sidecars gathered later)
for p in list_files(folders["lidar"]):
    if p.suffix.lower() != ".shp":
        continue
    m = _LIDAR_RE.match(p.stem)
    serial = f"p{int(m.group('serial')):05d}" if m else extract_serial(p.name)
    if serial:
        patches[serial]["lidar"] = p

# GT / HMA visual
for p in list_files(folders["gt"]):
    if p.suffix.lower() not in (".tif", ".tiff"):
        continue

    m = re.search(r"_p(\d+)(?=\.[^.]+$|$)", p.name, re.IGNORECASE)
    if not m:
        continue

    serial = f"p{int(m.group(1)):05d}"
    patches[serial]["gt"] = p

serials = sorted(patches.keys(), key=lambda s: int(s[1:]))
print(f"\nIndexed {len(serials)} unique patch serials (pXXXXX)")

complete = partial = 0
for s in serials:
    have = sum(1 for v in patches[s].values() if v is not None)
    if have == 4:
        complete += 1
    else:
        partial += 1
print(f"  complete (4/4): {complete}")
print(f"  partial  (<4):  {partial}")

if serials:
    demo = serials[0]
    print(f"\nExample {demo}:")
    for k, v in patches[demo].items():
        print(f"  {k:5s}: {v.name if v else '-'}")


Checking input folders:
  [OK] fab10  D:\Vaibhav\dem-lidar\inference_old\fabdem_10m_bicubic  (70 files)
  [OK] fab30  D:\Vaibhav\dem-lidar\inference_old\fabdem_30m  (71 files)
  [OK] lidar  D:\Vaibhav\dem-lidar\inference_old\lidar_shp  (350 files)
  [OK] gt     D:\Vaibhav\dem-lidar\gt_visual_256  (142 files)

Indexed 70 unique patch serials (pXXXXX)
  complete (4/4): 70
  partial  (<4):  0

Example p00000:
  fab10: 468_FABDEM_10mbicubic_p00000.tif
  fab30: 468_FABDEM_30m_p00000.tif
  lidar: 468_LiDAR_p00000.shp
  gt   : 00468_HMA_DEM8m_AT_20140225_0527_102001002B3B5E00_102001002B797800_DTM_10m_EGM08_p00000.tif


## 5. Build per-patch folders


In [5]:
output_root.mkdir(parents=True, exist_ok=True)

stats = {
    "folders": 0,
    "files": 0,
    "skipped_partial": 0,
    "missing_counts": defaultdict(int),
    "errors": 0,
}


def lidar_sidecars(shp_path: Path):
    """Yield all existing shapefile components next to the .shp."""
    seen = set()
    for ext in SHP_SIDECARS:
        cand = shp_path.with_suffix(ext)
        if cand.is_file():
            rp = cand.resolve()
            if rp not in seen:
                seen.add(rp)
                yield cand
    # catch oddities like foo.shp.xml
    for cand in shp_path.parent.glob(shp_path.stem + ".*"):
        if not cand.is_file():
            continue
        rp = cand.resolve()
        if rp in seen:
            continue
        low = cand.name.lower()
        if any(low.endswith(ext) for ext in SHP_SIDECARS) or low.endswith(".shp.xml"):
            seen.add(rp)
            yield cand


for serial in serials:
    rec = patches[serial]
    present = {k: v for k, v in rec.items() if v is not None}
    missing = [k for k, v in rec.items() if v is None]

    if not present:
        continue

    if missing and not ALLOW_PARTIAL:
        stats["skipped_partial"] += 1
        for m in missing:
            stats["missing_counts"][m] += 1
        print(f"skip {serial}: missing {missing}")
        continue

    patch_dir = output_root / serial
    patch_dir.mkdir(parents=True, exist_ok=True)
    stats["folders"] += 1

    for m in missing:
        stats["missing_counts"][m] += 1

    try:
        # 1) FABDEM 10 m — keep original name
        if rec["fab10"] is not None:
            src = rec["fab10"]
            transfer(src, patch_dir / src.name, COPY_MODE)
            stats["files"] += 1

        # 2) FABDEM 30 m — keep original name
        if rec["fab30"] is not None:
            src = rec["fab30"]
            transfer(src, patch_dir / src.name, COPY_MODE)
            stats["files"] += 1

        # 3) LiDAR shapefile + sidecars — keep original basenames
        if rec["lidar"] is not None:
            for src in lidar_sidecars(rec["lidar"]):
                transfer(src, patch_dir / src.name, COPY_MODE)
                stats["files"] += 1

        # 4) HMA / GT visual — simplified name
        if rec["gt"] is not None:
            src = rec["gt"]
            new_name = simplify_hma_name(src.name)
            transfer(src, patch_dir / new_name, COPY_MODE)
            stats["files"] += 1

        have_n = len(present)
        miss_txt = f"  missing={missing}" if missing else ""
        print(f"OK {serial}/  ({have_n}/4 products){miss_txt}")

    except Exception as e:
        stats["errors"] += 1
        print(f"ERROR {serial}: {e}")

print("\n==================================================")
print("              ARRANGE BY PATCH DONE              ")
print("==================================================")
print(f"Patch folders created: {stats['folders']:,}")
print(f"Files transferred:     {stats['files']:,}")
print(f"Skipped (partial):     {stats['skipped_partial']:,}")
print(f"Errors:                {stats['errors']:,}")
if stats["missing_counts"]:
    print("Missing product counts across patches:")
    for k, v in sorted(stats["missing_counts"].items()):
        print(f"  {k}: {v}")
print(f"\nOutput: {output_root}")


OK p00000/  (4/4 products)
OK p00001/  (4/4 products)
OK p00002/  (4/4 products)
OK p00003/  (4/4 products)
OK p00004/  (4/4 products)
OK p00005/  (4/4 products)
OK p00006/  (4/4 products)
OK p00007/  (4/4 products)
OK p00008/  (4/4 products)
OK p00009/  (4/4 products)
OK p00010/  (4/4 products)
OK p00011/  (4/4 products)
OK p00012/  (4/4 products)
OK p00013/  (4/4 products)
OK p00014/  (4/4 products)
OK p00015/  (4/4 products)
OK p00016/  (4/4 products)
OK p00017/  (4/4 products)
OK p00018/  (4/4 products)
OK p00019/  (4/4 products)
OK p00020/  (4/4 products)
OK p00021/  (4/4 products)
OK p00022/  (4/4 products)
OK p00023/  (4/4 products)
OK p00024/  (4/4 products)
OK p00025/  (4/4 products)
OK p00026/  (4/4 products)
OK p00027/  (4/4 products)
OK p00028/  (4/4 products)
OK p00029/  (4/4 products)
OK p00030/  (4/4 products)
OK p00031/  (4/4 products)
OK p00032/  (4/4 products)
OK p00033/  (4/4 products)
OK p00034/  (4/4 products)
OK p00035/  (4/4 products)
OK p00036/  (4/4 products)
O

## 6. Optional — list one patch folder


In [6]:
# Peek at the first patch folder
if output_root.is_dir():
    subs = sorted(
        [p for p in output_root.iterdir() if p.is_dir()],
        key=lambda p: int(p.name[1:]) if p.name.startswith("p") and p.name[1:].isdigit() else 10**12,
    )
    if not subs:
        print("No patch folders yet — run the previous cell.")
    else:
        sample = subs[0]
        print(f"Sample folder: {sample}")
        for f in sorted(sample.iterdir()):
            print(f"  {f.name:60s}  {f.stat().st_size/1024:8.1f} KB")
else:
    print("OUTPUT_ROOT does not exist yet.")


Sample folder: D:\Vaibhav\dem-lidar\inference\p00000
  00468_HMA_DTM_10m_EGM08_p00000.tif                             18509.0 KB
  468_FABDEM_10mbicubic_p00000.tif                               16633.0 KB
  468_FABDEM_30m_p00000.tif                                       1919.9 KB
  468_LiDAR_p00000.cpg                                               0.0 KB
  468_LiDAR_p00000.dbf                                              28.0 KB
  468_LiDAR_p00000.prj                                               0.4 KB
  468_LiDAR_p00000.shp                                              12.9 KB
  468_LiDAR_p00000.shx                                               3.8 KB


In [7]:
from pathlib import Path
import re

gt = Path(GT_ROOT)

print("GT folder exists:", gt.exists())
print("GT folder:", gt)

files = list(gt.glob("*"))

print("Number of files:", len(files))

for f in files[:10]:
    print(f.name)
    m = re.search(r"_p(\d+)", f.name)
    print("  match:", m.group(1) if m else None)

GT folder exists: True
GT folder: D:\Vaibhav\dem-lidar\gt_visual_256
Number of files: 142
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00070.tif
  match: 00070
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00070.tif.aux.xml
  match: 00070
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00071.tif
  match: 00071
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00071.tif.aux.xml
  match: 00071
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.tif
  match: 00072
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00072.tif.aux.xml
  match: 00072
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00073.tif
  match: 00073
00000_HMA_DEM8m_AT_20150317_0630_102001003D7C5600_102001003A13C500_DTM_10m_EGM08_p00073.tif.aux.xml
  match: 00073
00000_HMA_DEM8m_AT_201